# 0. Environment

In [1]:
import os
import pandas as pd
import csv
import json
from pathlib import Path
from collections import defaultdict, Counter
import re
import unicodedata

import time

In [2]:
os.chdir("C:\\Users\\yatsyk\\Documents\\NextCloud_Synchro\\passim_BoH\\")

# 1. Get familiar with *original* JSON structure

In [42]:
# get the keys
with open(r"C:\Users\yatsyk\deep-jsonl\parts_0006.jsonl", "r", encoding="utf-8") as f:
    line = f.readline()
    obj = json.loads(line)
    print(obj.keys())


dict_keys(['biblissima_id', 'iiif_manifest', 'notBefore', 'notAfter', 'shelfmark', 'tokens', 'language_source', 'language_normalized', 'language_recategorized', 'language_categorized', 'ark_portail', 'langue', 'notes_scopecontent', 'scopecontent', 'collections', 'languages_bbma', 'Unnamed: 8', 'seealso', 'files', 'scores'])


In [44]:
# get biblissima ids from & file
with open(r"C:\Users\yatsyk\deep-jsonl\\parts_0006.jsonl", "r", encoding="utf-8") as f:
    line = f.readline()
    obj = json.loads(line)
    print(obj.get("biblissima_id"))

https://data.biblissima.fr/entity/Q64711


# 2. Extract 196 mss

In [3]:
jsonl_dir = Path(r"C:\Users\yatsyk\deep-jsonl")
#csv_path = Path(r"04.CoMMa\02.metadata\livres-d'heures-comma-horae.csv")
csv_path = Path(r"04.CoMMa\02.metadata\196_arkindex_Comma_URL_filled.csv")

In [8]:
df = pd.read_csv(csv_path, sep=",", encoding="utf-8")
df = df[df["arkindex_comma_aligned"] == True]
df

,Manuscript H-ID,bbma_data_entity,ID Arkindex,URL CoMMA,Structural annotation,partially annotated,Leroquais_annotation,arkindex_comma_aligned
0,1356,https://data.biblissima.fr/entity/Q241713,6150a30a-bff0-4672-96c6-9c19896cdc5e,https://comma.inria.fr/doc/https%253A%252F%252...,False,False,False,True
1,1429,https://data.biblissima.fr/entity/Q235097,77fcbfe1-faf8-4316-8f32-0155ed3a60c5,https://comma.inria.fr/doc/https%253A%252F%252...,False,True,False,True
2,1430,https://data.biblissima.fr/entity/Q235098,034c35ae-17e3-4e51-9821-8cf1662b1253,https://comma.inria.fr/doc/https%253A%252F%252...,False,True,False,True
3,1642,https://data.biblissima.fr/entity/Q235484,73e26638-6275-41b9-9502-39e27f01d93f,https://comma.inria.fr/doc/https%253A%252F%252...,False,False,False,True
4,1829,https://data.biblissima.fr/entity/Q203702,b3b9759b-56e1-460d-b766-8cee25d6aaa5,https://comma.inria.fr/doc/https%253A%252F%252...,False,True,False,True
...,...,...,...,...,...,...,...,...
190,2761,https://data.biblissima.fr/entity/Q77215,60c22da0-28b5-4176-8734-c966db03cafc,https://comma.inria.fr/doc/https%253A%252F%252...,False,False,False,True
191,3118,https://data.biblissima.fr/entity/Q209026,5fe2eace-2177-4fa0-8c84-2b94b3d88c00,https://comma.inria.fr/doc/https%253A%252F%252...,False,True,False,True
192,3119,https://data.biblissima.fr/entity/Q209044,a9a01b20-b6cc-43e1-b3cd-4b57bb3fc270,https://comma.inria.fr/doc/https%253A%252F%252...,False,True,False,True
193,3355,https://data.biblissima.fr/entity/Q209755,c0b8fea9-df2b-4518-a55e-5337fa54ce13,https://comma.inria.fr/doc/https%253A%252F%252...,False,True,False,True


## 2.1. Extract

In [9]:
output_dir = Path(r"04.CoMMa\00.data\00.jsonls")
output_dir.mkdir(parents=True, exist_ok=True)

# read the biblissima ids from csv
df = pd.read_csv(csv_path, sep=",", encoding="utf-8")

bbma_ids = set(
    df["bbma_data_entity"]
    .dropna()
    .astype(str)
    .str.strip()
)

print(f"✅ bbma_data_entity: {len(bbma_ids)} entries")


def extract_short_id(full_id: str) -> str:
    """Extract short id from biblissima_id URL."""
    return full_id.rstrip("/").split("/")[-1]


# existing output files
existing_files = {p.stem for p in output_dir.glob("*.jsonl")}
print(f"Already existing files: {len(existing_files)}")

# ids for which output files already exist
ids_to_skip = {
    full_id for full_id in bbma_ids
    if extract_short_id(full_id) in existing_files
}

# ids that still need to be searched and written
ids_to_find = bbma_ids - ids_to_skip

print(f"IDs skipped because file already exists: {len(ids_to_skip)}")
print(f"IDs to search: {len(ids_to_find)}")


# cache open files
open_files = {}

found_ids = set()
created_ids = set()


def get_file_handle(full_id: str):
    short_id = extract_short_id(full_id)
    file_path = output_dir / f"{short_id}.jsonl"

    if short_id not in open_files:
        print(f"Creating new file: {file_path.name}")
        open_files[short_id] = open(file_path, "w", encoding="utf-8")
        created_ids.add(full_id)

    return open_files[short_id]


# go through jsonl files and extract matching objects
jsonl_files = list(jsonl_dir.rglob("*.jsonl"))

processed = 0
candidate_lines = 0
matched = 0

start = time.time()

for file in jsonl_files:
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            processed += 1

            # fast pre-filter: skip JSON parsing if none of the target ids
            
            id_pattern = re.compile(
                "|".join(re.escape(bid) for bid in ids_to_find)
            )

            if not id_pattern.search(line):
                continue

            candidate_lines += 1

            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            biblissima_id = obj.get("biblissima_id")
            if not biblissima_id:
                continue

            biblissima_id = str(biblissima_id).strip()

            if biblissima_id in ids_to_find:
                matched += 1
                found_ids.add(biblissima_id)

                out_f = get_file_handle(biblissima_id)
                out_f.write(json.dumps(obj, ensure_ascii=False) + "\n")

            if processed % 100000 == 0:
                elapsed = time.time() - start
                print(
                    print(f"{processed:,} lines processed in {elapsed/60:.1f} min"),
                    f"{candidate_lines} candidate lines | "
                    f"{matched} matches | "
                    f"{elapsed:.1f}s"
                )


# close all open files
for f in open_files.values():
    f.close()

missing_ids = ids_to_find - found_ids

print("\nDone!")
print(f"IDs requested: {len(bbma_ids)}")
print(f"IDs skipped because file already existed: {len(ids_to_skip)}")
print(f"IDs searched: {len(ids_to_find)}")
print(f"Processed lines: {processed}")
print(f"Candidate lines parsed as JSON: {candidate_lines}")
print(f"Matched rows: {matched}")
print(f"Unique matched IDs: {len(found_ids)}")
print(f"Created files: {len(created_ids)}")
print(f"Missing IDs among searched IDs: {len(missing_ids)}")

✅ bbma_data_entity: 196 entries
Already existing files: 196
IDs skipped because file already exists: 196
IDs to search: 0


KeyboardInterrupt: 

In [ ]:
all_ids_in_json = set()

for file in jsonl_files:
    with open(file, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i > 10000:
                break
            obj = json.loads(line)
            bid = obj.get("biblissima_id")
            if bid:
                all_ids_in_json.add(bid)

print("Sample JSON IDs:", list(all_ids_in_json)[:10])
print("Intersection:", len(all_ids_in_json & ids_to_find))

## 2.2. Check whether the encoding is ok

In [18]:
jsonl_dir = Path(r"04.CoMMa\00.data\00.jsonls")

pua_counter = Counter()
examples = {}

total_lines = 0
lines_with_pua = 0

for jsonl_path in jsonl_dir.rglob("*.jsonl"):

    print(f"Checking {jsonl_path.name}")

    with open(jsonl_path, "r", encoding="utf-8", errors="strict") as f:

        for line_number, line in enumerate(f, start=1):

            total_lines += 1

            found_in_line = False

            for ch in line:
                code = ord(ch)

                # Unicode Private Use Area
                if 0xE000 <= code <= 0xF8FF:

                    pua_counter[ch] += 1

                    if ch not in examples:
                        examples[ch] = {
                            "file": str(jsonl_path),
                            "line_number": line_number,
                            "snippet": line[:300]
                        }

                    found_in_line = True

            if found_in_line:
                lines_with_pua += 1

print("\n=== SUMMARY ===")
print(f"Total lines checked: {total_lines:,}")
print(f"Lines containing PUA symbols: {lines_with_pua:,}")

print("\n=== PUA SYMBOLS FOUND ===\n")

for ch, count in pua_counter.most_common():

    code = hex(ord(ch))

    print(f"Symbol: {repr(ch)}")
    print(f"Unicode: {code}")
    print(f"Count: {count}")

    ex = examples[ch]

    print(f"Example file: {ex['file']}")
    print(f"Example line: {ex['line_number']}")
    print(f"Snippet: {ex['snippet'][:200]}")
    print("-" * 60)

Checking Q106117.jsonl
Checking Q106514.jsonl
Checking Q106532.jsonl
Checking Q106533.jsonl
Checking Q106537.jsonl
Checking Q106540.jsonl
Checking Q106541.jsonl
Checking Q106543.jsonl
Checking Q106545.jsonl
Checking Q106616.jsonl
Checking Q106852.jsonl
Checking Q107627.jsonl
Checking Q107628.jsonl
Checking Q107656.jsonl
Checking Q107662.jsonl
Checking Q109291.jsonl
Checking Q109292.jsonl
Checking Q109588.jsonl
Checking Q109589.jsonl
Checking Q109592.jsonl
Checking Q109594.jsonl
Checking Q109595.jsonl
Checking Q109597.jsonl
Checking Q109599.jsonl
Checking Q109600.jsonl
Checking Q109622.jsonl
Checking Q109835.jsonl
Checking Q109836.jsonl
Checking Q118363.jsonl
Checking Q203702.jsonl
Checking Q203810.jsonl
Checking Q207683.jsonl
Checking Q208917.jsonl
Checking Q208936.jsonl
Checking Q209025.jsonl
Checking Q209026.jsonl
Checking Q209027.jsonl
Checking Q209028.jsonl
Checking Q209044.jsonl
Checking Q209189.jsonl
Checking Q209191.jsonl
Checking Q209192.jsonl
Checking Q209193.jsonl
Checking Q2

Symbol: '\uf1ac' is a Private Use Symbol. It won't be properly visualised by any tool, but we use UTF-8 and we preserve it on all the steps of data treatment.

# 3. Get familiar with *exported* JSONL structure. List all line types

## 3.1. Only line types - 196 mss

In [ ]:
jsonl_dir = Path("04.CoMMa\\00.data\\00.jsonls")

line_type_counter = Counter()

for jsonl_path in jsonl_dir.glob("*.jsonl"):

    try:
        with open(jsonl_path, "r", encoding="utf-8") as f:

            for line_number, line in enumerate(f, start=1):

                line = line.strip()

                if not line:
                    continue

                try:
                    record = json.loads(line)

                    files = record.get("files", [])

                    if not isinstance(files, list):
                        continue

                    for page in files:
                        zones = page.get("zones", [])

                        if not isinstance(zones, list):
                            continue

                        for zone in zones:
                            lines = zone.get("lines", [])

                            if not isinstance(lines, list):
                                continue

                            for line_obj in lines:
                                line_type = line_obj.get("type", "NO_TYPE")
                                line_type_counter[line_type] += 1

                except Exception as e:
                    print(
                        f"JSON error in {jsonl_path.name}, "
                        f"line {line_number}: {e}"
                    )

    except Exception as e:
        print(f"File error in {jsonl_path.name}: {e}")

print("\nLine types found:\n")

for line_type, count in line_type_counter.most_common():
    print(f"{line_type}: {count}")


Line types found:

DefaultLine: 1051234
HeadingLine: 57950
DropCapitalLine: 16070
InterlinearLine: 226
MusicLine: 34


## 3.2. Line types and corresponding zone types - 196 mss

In [ ]:
zone_line_counter = Counter()

for jsonl_path in jsonl_dir.glob("*.jsonl"):

    with open(jsonl_path, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            files = record.get("files", [])

            for page in files:

                zones = page.get("zones", [])

                for zone in zones:

                    zone_type = zone.get("type", "NO_ZONE_TYPE")

                    lines = zone.get("lines", [])

                    for line_obj in lines:

                        line_type = line_obj.get("type", "NO_LINE_TYPE")

                        zone_line_counter[(zone_type, line_type)] += 1

print("\nZone type → Line type:\n")

for (zone_type, line_type), count in zone_line_counter.most_common():
    print(f"{zone_type} → {line_type}: {count}")


Zone type → Line type:

MainZone → DefaultLine: 993571
MainZone → HeadingLine: 46639
NumberingZone → DefaultLine: 18718
text → DefaultLine: 18516
MainZone → DropCapitalLine: 12621
TitlePageZone → DefaultLine: 7186
MarginTextZone → DefaultLine: 6905
TitlePageZone → HeadingLine: 4621
text → HeadingLine: 3571
GraphicZone → DefaultLine: 3121
DropCapitalZone → DropCapitalLine: 2881
MarginTextZone → HeadingLine: 2070
DropCapitalZone → DefaultLine: 1955
GraphicZone → HeadingLine: 780
DigitizationArtefactZone → DefaultLine: 554
GraphicZone → DropCapitalLine: 495
RunningTitleZone → DefaultLine: 447
MainZone → InterlinearLine: 210
DropCapitalZone → HeadingLine: 208
QuireMarksZone → DefaultLine: 152
StampZone → DefaultLine: 75
text → DropCapitalLine: 58
RunningTitleZone → HeadingLine: 41
MusicZone → DefaultLine: 33
MainZone → MusicLine: 21
DigitizationArtefactZone → HeadingLine: 11
text → InterlinearLine: 11
StampZone → DropCapitalLine: 10
text → MusicLine: 7
QuireMarksZone → HeadingLine: 3
Stam

## 3.3. Only line types - 178 mss we can match

In [10]:
jsonl_dir = Path("04.CoMMa\\00.data\\00.jsonls")

# Get the list of target files from the CSV
target_files = set(
    df.loc[df["arkindex_comma_aligned"], "bbma_data_entity"]
      .dropna()
      .astype(str)
      .str.rsplit("/", n=1)
      .str[-1]
      .add(".jsonl")
)

print(f"Need to process {len(target_files)} files")

line_type_counter = Counter()

for jsonl_path in jsonl_dir.glob("*.jsonl"):

    if jsonl_path.name not in target_files:
        continue

    try:
        with open(jsonl_path, "r", encoding="utf-8") as f:

            for line_number, line in enumerate(f, start=1):

                line = line.strip()

                if not line:
                    continue

                try:
                    record = json.loads(line)

                    files = record.get("files", [])

                    if not isinstance(files, list):
                        continue

                    for page in files:
                        zones = page.get("zones", [])

                        if not isinstance(zones, list):
                            continue

                        for zone in zones:
                            lines = zone.get("lines", [])

                            if not isinstance(lines, list):
                                continue

                            for line_obj in lines:
                                line_type = line_obj.get("type", "NO_TYPE")
                                line_type_counter[line_type] += 1

                except Exception as e:
                    print(
                        f"JSON error in {jsonl_path.name}, "
                        f"line {line_number}: {e}"
                    )

    except Exception as e:
        print(f"File error in {jsonl_path.name}: {e}")

print("\nLine types found:\n")

for line_type, count in line_type_counter.most_common():
    print(f"{line_type}: {count}")

Need to process 178 files

Line types found:

DefaultLine: 975679
HeadingLine: 56207
DropCapitalLine: 15519
InterlinearLine: 202
MusicLine: 34


# 4. Extract transcriptions from jsonl

In [61]:
input_dir = Path(r"04.CoMMa\00.data\00.jsonls")
output_dir = Path(r"04.CoMMa\00.data\01.csvs_lineTypes")
output_dir.mkdir(exist_ok=True)

In [62]:
folio_pattern = re.compile(r'^[^-]+-(.+)\.xml$')
page_nmb_pattern = re.compile(r'^f(\d+)-')

## 4.1. Helper functions

### 4.1.1. Extract page metadata

In [51]:
def extract_metadata(filename: str) -> tuple[str | None, int | None]:
    """
    Extract:
    - folio: everything between the first "-" and ".xml"
    - CoMMA_transcribed_page_nmb: number after initial "f"
    """

    if not filename:
        return None, None

    filename = str(filename)

    # Extract folio
    folio_match = folio_pattern.search(filename)
    folio = folio_match.group(1) if folio_match else None

    # Extract page number
    page_match = page_nmb_pattern.search(filename)
    page_nmb = int(page_match.group(1)) if page_match else None

    return folio, page_nmb

### 4.1.2. Extract text from DefaultLines for each page

In [63]:
def extract_DefaultLines_from_record(record: dict) -> list[dict]:
    rows = []

    files = record.get("files", [])
    if not isinstance(files, list):
        return rows

    for page in files:
        page_id = page.get("alto", "")
        folio, page_nmb = extract_metadata(page_id)

        text_parts = []
        zones = page.get("zones", [])

        if isinstance(zones, list):
            for zone in zones:
                if zone.get("type") != "MainZone":
                    continue

                lines = zone.get("lines", [])
                if not isinstance(lines, list):
                    continue

                for line in lines:
                    if line.get("type") == "DefaultLine":
                        content = line.get("content", "")
                        if content is not None:
                            content = str(content).strip()
                            content = unicodedata.normalize("NFC", content)
                            if content:
                                text_parts.append(content)

        rows.append({
            "page_id": page_id,
            "CoMMA_folio_name": folio,
            "CoMMA_transcribed_page_nmb": page_nmb,
            "text": " ".join(text_parts)
        })

    return rows

### 4.1.3. Extract text from all line types

In [64]:
def extract_page_rows_from_record(record: dict) -> list[dict]:
    rows = []

    files = record.get("files", [])
    if not isinstance(files, list):
        return rows

    for page in files:
        page_id = page.get("alto", "")
        folio, page_nmb = extract_metadata(page_id)

        page_rows = []

        zones = page.get("zones", [])
        if isinstance(zones, list):
            for zone_index, zone in enumerate(zones):
                if not isinstance(zone, dict):
                    continue

                zone_type = zone.get("type", "")

                lines = zone.get("lines", [])
                if not isinstance(lines, list):
                    continue

                for line_index, line in enumerate(lines):
                    if not isinstance(line, dict):
                        continue

                    line_type = line.get("type", "")

                    content = line.get("content", "")
                    if content is None:
                        content = ""

                    content = str(content).strip()
                    content = unicodedata.normalize("NFC", content)

                    page_rows.append({
                        "page_id": page_id,
                        "CoMMA_folio_name": folio,
                        "CoMMA_transcribed_page_nmb": page_nmb,
                        "zone_index": zone_index,
                        "zone_type": zone_type,
                        "line_index": line_index,
                        "line_type": line_type,
                        "text": content,
                    })

        if page_rows:
            rows.extend(page_rows)
        else:
            rows.append({
                "page_id": page_id,
                "CoMMA_folio_name": folio,
                "CoMMA_transcribed_page_nmb": page_nmb,
                "zone_index": "",
                "zone_type": "",
                "line_index": "",
                "line_type": "",
                "text": "",
            })

    return rows

### 4.1.4. Transform jsonls to csvs

In [67]:
def process_jsonl_file(jsonl_path: Path, output_dir: Path) -> None:
    all_rows = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"JSON error in {jsonl_path.name}, line {line_num}: {e}")
                continue

            page_rows = extract_page_rows_from_record(record)
            all_rows.extend(page_rows)

    df = pd.DataFrame(all_rows, columns=["page_id", "CoMMA_folio_name", "CoMMA_transcribed_page_nmb", "zone_index", "zone_type", "line_index", "line_type", "text"])

    out_path = output_dir / f"{jsonl_path.stem}.csv"
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"✅ Saved: {out_path} ({len(df)} lines)")

## 4.2. Implementation

In [68]:
jsonl_files = list(input_dir.rglob("*.jsonl"))
print(f"Found jsonl files: {len(jsonl_files)}")

for jsonl_file in jsonl_files:
    process_jsonl_file(jsonl_file, output_dir)

print("Done.")

Found jsonl files: 196
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q106117.csv (3223 lines)
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q106514.csv (5690 lines)
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q106532.csv (3712 lines)
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q106533.csv (7060 lines)
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q106537.csv (9733 lines)
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q106540.csv (8139 lines)
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q106541.csv (4091 lines)
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q106543.csv (6642 lines)
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q106545.csv (5337 lines)
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q106616.csv (14062 lines)
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q106852.csv (7771 lines)
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q107627.csv (4088 lines)
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q107628.csv (7661 lines)
✅ Saved: 04.CoMMa\00.data\01.csvs_lineTypes\Q107656.csv (11203 lines)
✅ Saved: 

## 4.4. From lines to pages

In [73]:
input_dir = Path(r"04.CoMMa\00.data\01.csvs_lineTypes")
output_dir = Path(r"04.CoMMa\00.data\02.csvs_pages")
output_dir.mkdir(parents=True, exist_ok=True)

needed_cols = [
    "page_id",
    "CoMMA_folio_name",
    "CoMMA_transcribed_page_nmb",
    "zone_index",
    "zone_type",
    "line_index",
    "line_type",
    "text",
]

final_cols = [
    "page_id",
    "CoMMA_folio_name",
    "CoMMA_transcribed_page_nmb",
    "text",
]

for csv_path in sorted(input_dir.glob("*.csv")):
    df = pd.read_csv(csv_path, dtype=str, keep_default_na=False)

    missing_cols = [col for col in needed_cols if col not in df.columns]
    if missing_cols:
        print(f"Skipping {csv_path.name}: missing columns {missing_cols}")
        continue

    df = df[needed_cols].copy()

    # Preserve the order of pages as they appear in the original file
    page_order = (
        df[["page_id"]]
        .drop_duplicates()
        .reset_index(drop=True)
        .reset_index()
        .rename(columns={"index": "page_order"})
    )

    df = df.merge(page_order, on="page_id", how="left")

    # Convert technical indexes to numbers only for sorting
    df["zone_index_sort"] = pd.to_numeric(df["zone_index"], errors="coerce")
    df["line_index_sort"] = pd.to_numeric(df["line_index"], errors="coerce")

    df = df.sort_values(
        by=["page_order", "zone_index_sort", "line_index_sort"],
        na_position="last",
    )

    grouped = (
        df.groupby(
            [
                "page_order",
                "page_id",
                "CoMMA_folio_name",
                "CoMMA_transcribed_page_nmb",
            ],
            sort=False,
            dropna=False,
        )["text"]
        .apply(lambda s: "\n".join([value for value in s if value != ""]))
        .reset_index()
    )

    grouped = grouped.sort_values("page_order")[final_cols]

    output_path = output_dir / csv_path.name
    grouped.to_csv(output_path, index=False, encoding="utf-8-sig")

    print(f"Saved: {output_path}")

Saved: 04.CoMMa\00.data\02.csvs_pages\Q106117.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q106514.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q106532.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q106533.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q106537.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q106540.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q106541.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q106543.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q106545.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q106616.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q106852.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q107627.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q107628.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q107656.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q107662.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q109291.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q109292.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q109588.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q109589.csv
Saved: 04.CoMMa\00.data\02.csvs_pages\Q109592.csv


## 4.5. Add page_ordering (starting from 1)

In [74]:
csv_dir = output_dir

for csv_path in csv_dir.glob("*.csv"):
    df = pd.read_csv(csv_path)

    df["page_ordering"] = range(1, len(df) + 1)

    df.to_csv(csv_path, index=False)